# Regenerate all caches: TFM PCJ pipeline E2E

Orden DAG completo M03 -> M15 + building blocks Z03-Z06.

## Cómo usar

- **Ejecuta de arriba a abajo** para regen full desde 0 (~6h por M10 OBSO 25Hz).
- Cada celda es **autocontenida**: re-ejecutar una celda regenera solo ese módulo.
- Flag `FORCE` borra caches del módulo + regenera. Default `True`.
- Flag `RETRAIN` re-entrena modelos pesados (M04 SVI, M05 LGB+Optuna, M08 CatBoost+Optuna,
  Z03/Z04/Z06 LGB+Optuna, M11 SVI, M14 NUTS HMC). Default `True`.

## Topología (ejecutar en este orden)

```
M03 enrich
M04 WP                    [SVI bayes]
M05 PSxG → M05B           [LGB + Optuna 60 + isotonic]
M06 nearmiss
M07 shocks
M08 train atomic-VAEP     [CatBoost + Optuna 30, primer paso]
  ├── Z03 xpress          [tracking 25Hz + LGB]
  ├── Z04 vdep_strict     [Toda 2022 - 2 cabezas LGB]
  ├── Z05 maejima         [nearest-defender frame-level]
  └── Z06 un-xPass        [Robberechts 2023 - LGB]
M08 finalize              [per_minute + per_shock_window con un-xpass]
M09 defensa               [lee Z03/Z04/Z05 → score_def_v4]
M10 OBSO                  [CARO: ~4-5h con PPCF Z02 25Hz x 64 partidos]
M11 físico                [Bradley 2024 + SVI multivariate]
M12 DiD                   [TWFE-FE + Sun-Abraham + BJS + HonestDiD]
M12B validation           [placebo + power + window + stage]
M13 AIPW                  [DoubleMLIRM + DR-learner + RDD + spec]
M14 CATE                  [NUTS HMC 4 chains x 2k iter ~20 min]
M15 PCJ                   [tabla scout final]
```


In [ ]:
# === [0] Setup ============================================================
from pathlib import Path
import sys, time, shutil, warnings, os
import polars as pl

warnings.filterwarnings('ignore', category=UserWarning,   module='socceraction')
warnings.filterwarnings('ignore', category=FutureWarning, module='socceraction')
warnings.filterwarnings('ignore', category=DeprecationWarning, module='socceraction')
warnings.filterwarnings('ignore', category=FutureWarning, module='pandas')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='invalid value')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='divide by zero')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='overflow')
warnings.filterwarnings('ignore', category=UserWarning,   module='lightgbm')
warnings.filterwarnings('ignore', category=UserWarning,
                        message='X does not have valid feature names')
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jax')
warnings.filterwarnings('ignore', category=FutureWarning, module='jax')

os.environ['JAX_PLATFORMS'] = os.environ.get('JAX_PLATFORMS', 'cpu')
import logging
for log_name in ('numexpr', 'numpyro', 'numba', 'matplotlib'):
    logging.getLogger(log_name).setLevel(logging.WARNING)

candidates = [Path('.').resolve(), Path('..').resolve()]
REPO = next((p for p in candidates
             if (p / 'src' / 'M01_loader_pff.py').exists()), None)
assert REPO is not None, f'No encuentro REPO desde {Path.cwd()}'
SRC = REPO / 'src'
DERIVED = REPO / 'data' / 'parquet' / 'derived'
sys.path.insert(0, str(SRC))

print(f'REPO   : {REPO}')
print(f'DERIVED: {DERIVED}')


def _clean_dir(path: Path):
    if not path.exists(): return
    if path.is_dir(): shutil.rmtree(path)
    else: path.unlink()


def _clean_files(*paths: Path):
    for p in paths:
        if p.exists(): p.unlink()


In [ ]:
# === [1] Audit raw lossless ===============================================
# Verifica round-trip JSON original <-> parquet (no regenera nada).
SKIP_AUDIT = True   # cambia a False para correr (~3-5 min)

if SKIP_AUDIT:
    print('  AUDIT SKIPPED (SKIP_AUDIT=True). Cambia a False para ejecutar.')
else:
    from extract.audit import run_full_audit
    t0 = time.time()
    res = run_full_audit()
    print(f'  audit completo en {time.time()-t0:.1f}s, all_ok={res["all_ok"]}')


In [ ]:
# === [2] Clean derived caches =============================================
# Borra TODO derived/* + cache/vaep. Util para regen full desde 0.
# pff_grades.parquet se regenera en celda [3] (no se preserva).
FORCE_CLEAN = False   # PELIGRO: a True borra ~120MB, regen completa ~6h

if FORCE_CLEAN:
    _clean_dir(DERIVED)
    _clean_dir(REPO / 'cache' / 'vaep')
    print('  TODOS los caches derived borrados.')
else:
    print('  FORCE_CLEAN=False. Las celdas individuales gestionan su cache.')


In [ ]:
# === [3] pff_grades extract (preprocess para priors M14) ==================
# Outputs : derived/preprocess/pff_grades.parquet
# Consumes: PFF events.grades struct + rosters
# Re-run downstream: M14 attach_pff_grades
FORCE = True

import importlib, src.preprocess.pff_grades_extract as pff_grades
importlib.reload(pff_grades)

t0 = time.time()
out = pff_grades.main(overwrite=FORCE)
print(f'  pff_grades.parquet generado en {time.time()-t0:.0f}s -> {out}')
df = pl.read_parquet(out)
print(f'  {df.height} jugadores, mean grade {df["pff_grade_mean"].mean():+.4f}')


In [ ]:
# === [4] M03 preprocess - enrich_events ===================================
# Outputs : derived/preprocess/events_enriched/{match_id}.parquet x 64
# Consumes: M01 (PFF events + metadata) + M02 (SB matches + events para
#           goals_timeline con sgc real PFF)
# Re-run downstream: ninguno (events_enriched no se consume directamente)
FORCE = True

import M03_preprocess as m03

target_dir = DERIVED / 'preprocess' / 'events_enriched'
if FORCE:
    _clean_dir(target_dir)

t0 = time.time()
out = m03.cache_all_enriched(overwrite=FORCE)
print(f'  enrich_events: {len(out)} matches en {time.time()-t0:.0f}s')
sample_path = next(target_dir.glob('*.parquet'))
sample = pl.read_parquet(sample_path)
print(f'  sample {sample_path.name}: {sample.height:,} rows x {sample.width} cols')


In [ ]:
# === [5] M04 WP - SVI bayes regulation + temperature scaling + per_minute =
# Outputs : derived/wp/training/training_matrix.parquet
#           derived/wp/model/wp_regulation.pkl
#           derived/wp/per_minute.parquet (con leverage + elim_prox_home/away)
# Consumes: Wyscout events + SB events (no WC22)
# Re-run downstream: M07 (leverage_at_shock + elim_prox), M14, M15
FORCE = True
RETRAIN = True   # SVI ~3 min

import M04_wp as m04

if FORCE:
    if RETRAIN:
        _clean_files(m04._CACHE / 'training_matrix.parquet',
                     m04._MODEL / 'wp_regulation.pkl')
    _clean_files(m04._DERIVED / 'per_minute.parquet')

t0 = time.time()
X, info = m04.build_training_matrix(cache=True)
print(f'  training matrix: {info}')

fit_path = m04._MODEL / 'wp_regulation.pkl'
if fit_path.exists() and not RETRAIN:
    fit = m04.load_fit(fit_path)
else:
    X_tr, X_val = m04.train_val_split(X, val_frac=0.2, seed=42)
    fit = m04.fit_wp(X_tr, n_steps=3000)
    X_calib = m04.build_wc22_groups_calib_matrix()
    T_opt = m04.fit_temperature(fit, X_calib)
    fit['temperature'] = T_opt
    m04.save_fit(fit)
    print(f'  T optimo = {T_opt:.4f}')

group_ctx = m04.build_wc22_group_context()
out = m04.cache_all_wp(fit, overwrite=True, group_ctx=group_ctx)
big = pl.read_parquet(out)
print(f'  per_minute: {big.height:,} filas, {big["match_id"].n_unique()}/64 matches '
      f'en {time.time()-t0:.0f}s')


In [ ]:
# === [6] M05 PSxG - LightGBM + Optuna 60 + isotonic + apply WC22 ==========
# Outputs : derived/psxg/{training_shots,wc22_shots,shots}.parquet
#           derived/psxg/model/psxg_lgb.pkl
# Consumes: SB events + 360 freeze (Euro20+24+Bundes23 train, WC22 apply)
# Re-run downstream: M06 nearmiss, M10 (xg_grid), M13 (cov shots)
FORCE = True
RETRAIN = True   # Optuna 60 trials ~10-15 min

import M05_psxg as m05

if FORCE:
    if RETRAIN:
        _clean_files(m05._DERIVED / 'training_shots.parquet',
                     m05._DERIVED / 'wc22_shots.parquet',
                     m05._MODEL / 'psxg_lgb.pkl')
    _clean_files(m05._DERIVED / 'shots.parquet')

t0 = time.time()
train = m05.build_training_shots(cache=True)
print(f'  training shots: {train.height} ({int(train["_label"].sum())} goles)')

fit_path = m05._MODEL / 'psxg_lgb.pkl'
if fit_path.exists() and not RETRAIN:
    fit = m05.load_fit(fit_path)
else:
    fit = m05.fit_psxg(train, n_folds=5, seed=42, n_trials=60)
    m05.save_fit(fit)
    m = fit['metrics']
    print(f'  AUC OOF cal {m["auc_psxg_calibrated"]:.4f} '
          f'vs SB baseline {m["auc_baseline_sb_xg"]:.4f}')

out = m05.cache_wc22_psxg(fit, overwrite=True)
big = pl.read_parquet(out)
print(f'  shots WC22: {big.height} ({big["is_goal"].sum()} goles) '
      f'en {time.time()-t0:.0f}s')


In [ ]:
# === [7] M05B PSxG calibration plots ======================================
# Outputs : derived/psxg/calibration/{calibration_curve,brier_decomposition,
#                                      calibration_metrics,isotonic_curve}.parquet
# Consumes: M05 model + training_shots + wc22_shots
FORCE = True

if FORCE:
    _clean_dir(DERIVED / 'psxg' / 'calibration')

t0 = time.time()
import M05B_calibration as m05b
m05b.compute_all()
print(f'  M05B calibration plots en {time.time()-t0:.0f}s')


In [ ]:
# === [8] M06 nearmiss - 5 tipos near-miss desde PSxG WC22 =================
# Outputs : derived/nearmiss/nearmiss_table.parquet
# Consumes: M05 shots, SB events + 360 freeze, PFF tracking 25Hz (GLT)
# Re-run downstream: M13 AIPW
FORCE = True

import M06_nearmiss as m06

if FORCE:
    _clean_files(m06._DERIVED / 'nearmiss_table.parquet')

t0 = time.time()
nm = m06.build_near_miss_table(cache=True, overwrite=True)
print(f'  near-miss: {nm.height} en {time.time()-t0:.0f}s')
print(m06.summary_by_type(nm))


In [ ]:
# === [9] M07 shocks - 172 shocks-gol + ventanas +-10min por jugador ======
# Outputs : derived/shocks/shocks_table.parquet (sec_abs PFF real)
#           derived/shocks/shocks_team_members.parquet (composicion bloque LOO)
# Consumes: M01 PFF events + M03 goals_timeline + player_minutes + M04 leverage
# Re-run downstream: M08-M11 per_shock_window + LOO, M12, M14, M15
FORCE = True

import M07_shocks as m07

if FORCE:
    _clean_files(m07._DERIVED / 'shocks_table.parquet',
                 m07._DERIVED / 'shocks_team_members.parquet')

t0 = time.time()
sh = m07.build_shocks_table(cache=True, overwrite=True)
print(f'  shocks: {sh.height:,} rows ({sh["shock_id"].n_unique()} shocks unicos) '
      f'en {time.time()-t0:.0f}s')
print(m07.summary_shocks(sh))


In [ ]:
# === [10] M08 partA: train atomic-VAEP + apply WC22 (sin aggregate) ======
# Outputs : derived/ataque/{training_atomic,wc22_atomic}.parquet
#           derived/ataque/model/vaep_atk_*.cbm + meta.pkl
#           derived/ataque/sb_to_pff_player_map.parquet
# IMPORTANTE: per_minute + per_shock_window se generan en celda [15] M08 partB
#             (despues de Z03-Z06, para incluir un-xpass en el outcome v2).
FORCE = True
RETRAIN = True   # CatBoost + Optuna 30 ~30-45 min

import M08_ataque as atk

if FORCE:
    if RETRAIN:
        _clean_files(atk._DERIVED / 'training_atomic.parquet',
                     atk._DERIVED / 'wc22_atomic.parquet')
        _clean_dir(atk._MODEL_DIR)
        _clean_dir(REPO / 'cache' / 'vaep')
    _clean_files(atk._DERIVED / 'sb_to_pff_player_map.parquet',
                 atk._DERIVED / 'per_minute.parquet',
                 atk._DERIVED / 'per_shock_window.parquet')

t0 = time.time()
print('[1] atomic SPADL training (Euro20+Euro24+Bundes23)...')
train_df = atk.build_training_atomic(overwrite=RETRAIN)
print(f'  training atomic: {len(train_df):,} acciones')

print('[2] atomic SPADL WC22...')
wc22_df = atk.build_wc22_atomic(overwrite=RETRAIN)
print(f'  wc22 atomic: {len(wc22_df):,} acciones')

print('[3] VAEP model...')
fit_path = atk._MODEL_DIR / 'vaep_atk_meta.pkl'
if fit_path.exists() and not RETRAIN:
    fit = atk.load_models()
else:
    fit = atk.train_vaep_model(train_df, n_folds=5, seed=42, tune=True, n_trials=30)
    atk.save_models(fit)
    m = fit['metrics']
    print(f'  AUC scores OOF cal {m["auc_scores_oof_cal"]:.4f}')

print('[4] mapping SB -> PFF player_id...')
mapping = atk.build_sb_to_pff_player_map(cache=True)
mapped = mapping.filter(pl.col('pff_player_id').is_not_null()).height
print(f'  mapping: {mapped}/{mapping.height} matched ({100*mapped/mapping.height:.1f}%)')
print(f'  total partA: {time.time()-t0:.0f}s')


In [ ]:
# === [11] Z03 exPress - P(recovery<5s|press) Lee 2025 + tracking 25Hz =====
# Outputs : derived/defensa/xpress/{training,model,per_event,per_minute}.parquet
# Consumes: PFF events + tracking 25Hz (asof BACKWARD por videoTimeMs)
# Re-run downstream: M09 (xpress_value_minute en score_def_v4)
FORCE = True

import Z03_xpress as z03

if FORCE:
    _clean_dir(z03._DERIVED)

t0 = time.time()
paths = z03.compute_all(overwrite=True, n_trials=30)
print(f'  Z03 xpress en {time.time()-t0:.0f}s')


In [ ]:
# === [12] Z04 VDEP strict - Toda 2022 P(recovery)-C*P(attacked) ===========
# Outputs : derived/defensa/vdep_strict/{training,model,per_event,per_minute}.parquet
# Consumes: M08 atomic SPADL training (Euro20+24+Bundes23) + WC22 atomic
# Re-run downstream: M09 (vdep_strict_minute en score_def_v4)
FORCE = True

import Z04_vdep as z04

if FORCE:
    _clean_dir(z04._DERIVED)

t0 = time.time()
paths = z04.compute_all(overwrite=True, n_trials=25)
print(f'  Z04 vdep_strict en {time.time()-t0:.0f}s')


In [ ]:
# === [13] Z05 Maejima - nearest-defender frame-level attribution ==========
# Outputs : derived/defensa/maejima/{per_event,per_minute}.parquet
# Consumes: M08 model + apply WC22 + tracking PFF 25Hz
# Re-run downstream: M09 (maejima_value_minute en score_def_v4)
FORCE = True

import Z05_maejima as z05

if FORCE:
    _clean_dir(z05._DERIVED)

t0 = time.time()
paths = z05.compute_all(overwrite=True)
print(f'  Z05 maejima en {time.time()-t0:.0f}s')


In [ ]:
# === [14] Z06 un-xPass - Robberechts 2023 P(success) creative residual ===
# Outputs : derived/ataque/unxpass/{training,model,per_event,per_minute}.parquet
# Consumes: M08 atomic SPADL training + WC22 + VAEP applied
# Re-run downstream: M08 partB (score_atk_v2 = atomic-VAEP + un-xpass)
FORCE = True

import Z06_unxpass as z06

if FORCE:
    _clean_dir(z06._DERIVED)

t0 = time.time()
paths = z06.compute_all(overwrite=True, n_trials=25)
print(f'  Z06 un-xPass en {time.time()-t0:.0f}s')


In [ ]:
# === [15] M08 partB: aggregate per_minute + per_shock_window ============
# Ahora un-xpass YA existe -> score_atk_v2_minute incluye creative decision.
# LOO sobre score_atk_v2 + score_atk legacy (sensitivity).
import importlib
importlib.reload(atk)   # picks up un-xpass parquet path

t0 = time.time()
print('[1] apply VAEP a WC22...')
wc22_with_vaep = atk.apply_vaep_to_wc22(fit, wc22_df)

print('[2] aggregate per_minute (con un-xpass joinado)...')
per_min = atk.aggregate_per_player_minute(wc22_with_vaep, cache=True)
print(f'  per_minute: {per_min.height:,} filas')
print(f'  cols: {per_min.columns}')

print('[3] aggregate per_shock_window (LOO v2 + legacy)...')
per_shock = atk.aggregate_per_shock_window(per_min, mapping, cache=True)
print(f'  per_shock: {per_shock.height:,} filas')
print(f'  total M08 partB: {time.time()-t0:.0f}s')


In [ ]:
# === [16] M09 defensa - score_def_v4 = vdep_strict + xpress + maejima ====
# Outputs : derived/defensa/{press_value,def_third_context,per_minute,
#                            per_shock_window}.parquet
# Consumes: M08 model + apply, M07 shocks, tracking PFF 25Hz, Z03/Z04/Z05
# Re-run downstream: M12, M14, M15
FORCE = True

import M09_defensa as df9

if FORCE:
    _clean_files(df9._DERIVED / 'def_third_context.parquet',
                 df9._DERIVED / 'press_value.parquet',
                 df9._DERIVED / 'per_minute.parquet',
                 df9._DERIVED / 'per_shock_window.parquet')

t0 = time.time()
print('[1] press_value (Maejima light via PFF initialPressurePlayerId)...')
pv = df9.build_press_value_all(cache=True)
print(f'  press_value: {pv.height:,} rows')

print('[2] def_third + pressing context (tracking 25Hz x 64 matches)...')
ctx = df9.build_def_third_all(cache=True)
print(f'  context: {ctx.height:,} rows')

print('[3] per_minute (vdep_strict + xpress + maejima + press_value)...')
per_min = df9.aggregate_per_player_minute(cache=True)
print(f'  per_minute: {per_min.height:,} filas')
print(f'  cols: {per_min.columns}')

print('[4] per_shock_window (LOO v4/v3/v2/legacy)...')
per_shock = df9.aggregate_per_shock_window(cache=True)
print(f'  per_shock: {per_shock.height:,} filas')
print(f'  total: {time.time()-t0:.0f}s')


In [ ]:
# === [17] M10 OBSO Spearman 2018 + C-OBSO (CARO ~4-5h) ===================
# Outputs : derived/offball/{xg_grid.npy,per_minute,per_shock_window}.parquet
# Consumes: M05 training_shots (xG grid), tracking PFF 25Hz (Z02 PPCF), M08 player_map
# Re-run downstream: M12, M14, M15
# CARO: 25Hz full-quality x 64 matches via PPCF Z02 vectorizado
FORCE = True

import M10_offball as m10

if FORCE:
    _clean_files(m10._DERIVED / 'xg_grid.npy',
                 m10._DERIVED / 'per_minute.parquet',
                 m10._DERIVED / 'per_shock_window.parquet')

t0 = time.time()
print('[1] xG grid (de M05 training shots)...')
xg = m10.build_xg_grid(cache=True)
print(f'  xg_grid {xg.shape}: min={xg.min():.3f}, max={xg.max():.3f}')

print('[2] OBSO + C-OBSO (PPCF Z02 25Hz x 64 matches) — paciencia ~4-5h')
per_min = m10.aggregate_per_player_minute(cache=True)
print(f'  per_minute: {per_min.height:,} filas')

print('[3] per_shock_window (LOO c_obso)...')
per_shock = m10.aggregate_per_shock_window(cache=True)
print(f'  per_shock: {per_shock.height:,} filas')
print(f'  total: {time.time()-t0:.0f}s')


In [ ]:
# === [18] M11 fisico - Bradley 2024 + SVI multivariate residual ===========
# Outputs : derived/fisico/{raw_per_minute,per_minute,per_shock_window}.parquet
#           derived/fisico/model/phys_state.pkl
# Consumes: tracking PFF 25Hz, M07 shocks, M08 player_map
FORCE = True
RETRAIN = True

import M11_fisico as m11

if FORCE:
    if RETRAIN:
        _clean_files(m11._DERIVED / 'model' / 'phys_state.pkl')
    _clean_files(m11._DERIVED / 'raw_per_minute.parquet',
                 m11._DERIVED / 'per_minute.parquet',
                 m11._DERIVED / 'per_shock_window.parquet')

t0 = time.time()
print('[1] raw metrics Bradley 2024 (Butterworth + Hampel + segmentation)...')
raw = m11.build_raw_per_minute(cache=True, overwrite=FORCE)
print(f'  raw: {raw.height:,} filas')

print('[2] SVI multivariate bayes + score_phys (residual z-score)...')
pm = m11.cache_score_phys(overwrite=FORCE, n_steps=4000)
print(f'  per_minute: {pm.height:,} filas, '
      f'score_phys mean={pm["score_phys"].mean():+.4f} '
      f'std={pm["score_phys"].std():.4f}')

print('[3] per_shock_window (LOO score_phys)...')
per_shock = m11.aggregate_per_shock_window(cache=True)
print(f'  per_shock: {per_shock.height:,} filas')
print(f'  total: {time.time()-t0:.0f}s')


In [ ]:
# === [19] M12 DiD - within-player TWFE + Sun-Abraham + BJS + HonestDiD ===
# Outputs : derived/did/{panel_*,ate_population,event_study,honest_did,
#                        diagnostics,ate_with_controls}.parquet
# Consumes: M07 shocks + M08-M11 per_minute (4 canales)
# Re-run downstream: M13 (comparison_m12), M15
FORCE = True

import M12_did as did

if FORCE:
    _clean_dir(did._DERIVED)

t0 = time.time()
panels = did.build_all_panels(clean_only=True, cache=True)
for ch, p in panels.items():
    print(f'  {ch:<10} {p.height:>7,} rows ({p["shock_id"].n_unique()} shocks)')

paths = did.compute_all(cache=True, overwrite=True)
print(f'  M12 todos los outputs en {time.time()-t0:.0f}s')

ate = pl.read_parquet(paths['ate'])
print('\n  ATE 4 canales x 2 shock_types:')
print(ate.select(['channel','shock_type','ate','ci_lo','ci_hi','ate_bjs','n_obs']))


In [ ]:
# === [20] M12B validation - placebo + power + window + stage =============
# Outputs : derived/did_validation/{baseline_naive,placebo_test,power_analysis,
#                                    window_sensitivity,stage_stratified}.parquet
# Consumes: M12 panels + ate_population
# Re-run downstream: M15 (power flags)
FORCE = True

if FORCE:
    _clean_dir(DERIVED / 'did_validation')

t0 = time.time()
import M12B_validation as v12b
v12b.main(overwrite=True)
print(f'  M12B en {time.time()-t0:.0f}s')


In [ ]:
# === [21] M13 AIPW - cuasi-experimento near-miss + DoubleMLIRM ===========
# Outputs : derived/aipw/{panel_master,att_aipw,att_dml_plr,att_dr_learner,
#                          att_rdd,spec_curve,balance,sensitivity,
#                          comparison_m12}.parquet
# Consumes: M03 mappings, M05 shots+wc22_shots, M06 nearmiss, M07 shocks,
#           M08-M11 per_minute (4 canales SOTA), M12 ate_population
# Re-run downstream: M15
FORCE = True

import M13_aipw as aipw

if FORCE:
    _clean_dir(aipw._DERIVED)

t0 = time.time()
paths = aipw.compute_all(cache=True, overwrite=True)
print(f'  M13 todos los outputs en {time.time()-t0:.0f}s')

ate_aipw = pl.read_parquet(paths['att_aipw'])
print('\n  ATT AIPW (4 canales x 2 perspectives):')
print(ate_aipw.select(['channel','perspective','att','ci_lo','ci_hi','n_obs']))

cmp = pl.read_parquet(paths['comparison_m12'])
print(f'\n  same_sign M12 vs M13: {int(cmp["same_sign"].sum())}/{cmp.height}')


In [ ]:
# === [22] M14 CATE - jerarquico bayesiano multivariate (NUTS HMC) ========
# Outputs : derived/cate/{panel_delta,posterior_player,posterior_corr,
#                          indices,rankings,diagnostics,ppc}.parquet
#           derived/cate/model/cate_nuts.pkl
# Consumes: M07 shocks, M08-M11 per_shock_window (delta_relative LOO),
#           pff_grades.parquet (priors)
# Re-run downstream: M15
# CARO: NUTS HMC 4 chains x (1k warmup + 1k samples) ~20 min
FORCE = True

import M14_cate as cate

if FORCE:
    _clean_dir(cate._DERIVED)

t0 = time.time()
paths = cate.compute_all(cache=True, overwrite=True,
                          num_warmup=cate.NUTS_NUM_WARMUP,
                          num_samples=cate.NUTS_NUM_SAMPLES,
                          num_chains=cate.NUTS_NUM_CHAINS)
print(f'  M14 NUTS done en {time.time()-t0:.0f}s')

idx = pl.read_parquet(paths['indices'])
diag = pl.read_parquet(paths['diagnostics'])
n_conv = diag.filter(pl.col('converged')).height
print(f'  indices: {idx.height} jugadores')
print(f'  diagnostics: {n_conv}/{diag.height} params convergidos (R-hat<1.05 + ESS>400)')


In [ ]:
# === [23] M15 PCJ - tabla maestra scout final =============================
# Outputs : outputs/pcj_table.parquet (~120 cols)
#           outputs/pcj_aux/{top10_chasing_per_position,top10_protecting_per_position,
#                            top10_pressure_per_position,dual_clutch_top,
#                            by_team}.parquet
# Consumes: M14 (posterior + indices + samples), M12+M12B+M13 (credibility +
#           power), M07 shocks, M04 leverage, M06 nearmiss, M11 raw fisico,
#           M08-M11 per_minute (baselines)
FORCE = True

if FORCE:
    out_dir = REPO / 'outputs'
    if out_dir.exists():
        shutil.rmtree(out_dir)

t0 = time.time()
import M15_pcj as m15
m15.main()
print(f'  M15 en {time.time()-t0:.0f}s')

pcj = pl.read_parquet(REPO / 'outputs' / 'pcj_table.parquet')
print(f'  pcj_table: {pcj.height} jugadores x {pcj.width} cols')
print('\n  Top 5 Remontador:')
print(pcj.sort('rank_chasing_global').head(5).select(
    ['rank_chasing_global','player_name','team_name','position_group',
     'chasing_clutch_idx','p_chasing_positive','sig_chasing']))


In [ ]:
# === [24] Smoke final - listado + tamano de todos los caches =============
def _walk_size(base: Path):
    if not base.exists():
        return None, []
    files = []
    for ext in ('*.parquet', '*.pkl', '*.cbm', '*.npy'):
        files.extend(base.rglob(ext))
    files = sorted(set(files))
    return sum(f.stat().st_size for f in files) / 1024 / 1024, files

print(f'{"PATH":<60} {"SIZE_MB":>10}')
print('-' * 75)
total = 0.0
for sub in ['preprocess', 'wp', 'psxg', 'nearmiss', 'shocks',
            'ataque', 'ataque/unxpass',
            'defensa', 'defensa/xpress', 'defensa/vdep_strict', 'defensa/maejima',
            'offball', 'fisico', 'did', 'did_validation', 'aipw', 'cate']:
    base = DERIVED / sub
    size, files = _walk_size(base)
    if size is None:
        print(f'{str(sub):<60} {"":>10}  MISSING')
        continue
    total += size
    print(f'{str(sub):<60} {size:>10.2f}')
print('-' * 75)
print(f'{"TOTAL derived/":<60} {total:>10.2f} MB')

out_pcj = REPO / 'outputs' / 'pcj_table.parquet'
if out_pcj.exists():
    print(f'\noutputs/pcj_table.parquet: {out_pcj.stat().st_size//1024} KB')
